<a href="https://colab.research.google.com/github/andremarcelino-py/aidrive1/blob/main/2Exercicio_1_Rodada_2_Colab_Alunos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício 1 — Rodada 2  
## Testes unitários com IA: comportamento observado × especificação

Na **Rodada 1**, você recebeu apenas uma implementação e foi orientado a:

- observar o comportamento do código;
- não inventar regras de negócio;
- gerar testes de caracterização;
- registrar de onde veio o *oracle* de cada teste.

Agora uma nova informação será disponibilizada: **a especificação funcional**.

O objetivo desta rodada é responder a uma pergunta essencial em engenharia de software:

> **Uma suíte de testes que passa significa necessariamente que o software está correto?**

Ao final, você deverá conseguir distinguir:

- teste de caracterização;
- teste derivado de requisito;
- comportamento observado;
- comportamento esperado;
- defeito de implementação;
- *test oracle*;
- falso senso de segurança causado por testes que apenas reproduzem o código existente.


## 1. Preparação do ambiente

Execute a célula abaixo para garantir que o `pytest` esteja disponível no Google Colab.


In [1]:
!pip -q install pytest

## 2. Implementação recebida na Rodada 1

A implementação continua exatamente a mesma.

**Não altere o código ainda.**


In [2]:
%%writefile frete.py
def calcular_frete(valor_compra: float, cliente_premium: bool) -> float:
    """
    Calcula o valor do frete de uma compra.
    """
    if valor_compra < 0:
        raise ValueError("valor_compra não pode ser negativo")

    if cliente_premium and valor_compra > 200:
        return 0.0

    return 20.0


Writing frete.py


## 3. Sua suíte da Rodada 1

Cole abaixo a suíte de testes que você produziu na Rodada 1.

Ela deve caracterizar o comportamento da implementação **sem utilizar a especificação funcional**, pois ela ainda não havia sido fornecida.


In [5]:
%%writefile test_frete_rodada1.py


import pytest
from frete import calcular_frete

def test_frete_gratis_premium_acima_200():
    # Expectativa (0.0): Inferida da condição 'if cliente_premium and valor_compra > 200'
    assert calcular_frete(200.01, True) == 0.0

def test_frete_padrao_premium_exatamente_200():
    # Expectativa (20.0): A condição '> 200' é falsa para 200.0, caindo no 'return 20.0'
    assert calcular_frete(200.0, True) == 20.0

def test_frete_padrao_premium_abaixo_200():
    # Expectativa (20.0): A condição '> 200' é falsa, caindo no 'return 20.0'
    assert calcular_frete(199.99, True) == 20.0

def test_frete_padrao_nao_premium_acima_200():
    # Expectativa (20.0): A condição 'cliente_premium' é False, caindo no 'return 20.0'
    assert calcular_frete(250.0, False) == 20.0

def test_frete_padrao_limite_zero():
    # Expectativa (20.0): O valor 0.0 falha no teste de erro (< 0) e cai no 'return 20.0'
    assert calcular_frete(0.0, False) == 20.0

def test_erro_valor_negativo():
    # Expectativa (ValueError): Inferida da condição inicial 'if valor_compra < 0'
    with pytest.raises(ValueError, match="valor_compra não pode ser negativo"):
        calcular_frete(-0.01, True)


Overwriting test_frete_rodada1.py


### Execute a suíte da Rodada 1

Antes de conhecer a especificação, registre o resultado.

**Pergunta:** todos os testes passam?
Sim



In [6]:
!python -m pytest -q test_frete_rodada1.py

......                                                                   [100%]
6 passed in 0.01s


### Registro 1 — interpretação

Responda antes de continuar:

1. Se todos os testes da Rodada 1 passarem, o que isso prova?
prova apenas que o código faz exatamente o que está escrito nele. Se a suíte rodar toda verde, significa que a lógica interna está coerente e não quebrou ao ser executada. Mas isso não garante que essa lógica é a que o negócio realmente precisa
2. O resultado prova que a regra de negócio está correta?

não. como a IA gerou os testes olhando só para a implementação, ela acaba "comprando" até os erros do programador. Se tiver um bug de lógica — tipo um > onde deveria ser >= —, o teste vai achar que aquele erro é o comportamento certo e passar do mesmo jeito. O teste verde só mostra que o programa é fiel ao código atual, não às regras do sistema.

3. Qual era a principal fonte do *oracle* dos testes da Rodada 1?

o próprio código-fonte, como não tínhamos nenhuma especificação ou documento de requisitos para consultar, a IA tirou os resultados esperados direto das decisões (if) e retornos (return) da função calcular_frete



# 4. Nova informação: especificação funcional

A partir deste ponto, existe uma especificação independente da implementação.

## Especificação de `calcular_frete`

A função deve obedecer às seguintes regras:

1. `valor_compra` não pode ser negativo.  
   - Se `valor_compra < 0`, deve lançar `ValueError`.

2. Cliente **premium** com compra de **R$ 200,00 ou mais** deve ter frete grátis.  
   - Retorno esperado: `0.0`.

3. Nos demais casos, o frete deve custar **R$ 20,00**.  
   - Retorno esperado: `20.0`.

A partir de agora, essas regras podem ser utilizadas como fonte do *oracle* dos testes.


## 5. Antes de executar qualquer novo teste: compare código e requisito

Analise manualmente a implementação.

Preencha a tabela conceitualmente:

| Situação | Especificação | Implementação atual | Coincidem? |
|---|---:|---:|---|
| `valor_compra < 0` | ? | ? | ? |
| premium e `valor_compra = 199.99` | ? | ? | ? |
| premium e `valor_compra = 200.00` | ? | ? | ? |
| premium e `valor_compra = 200.01` | ? | ? | ? |
| não premium e `valor_compra = 500.00` | ? | ? | ? |

### Pergunta central

Existe algum valor de entrada para o qual:

- os testes de caracterização da Rodada 1 podem estar corretos;
- a implementação pode continuar fazendo exatamente o que fazia;
- e, ainda assim, a especificação seja violada?

**Sua resposta:**

>
Sim, o valor  é 200 para clientes premium, na Rodada 1, o teste para o valor de 200 esperava como retorno 20.0 porque foi montado olhando só a lógica do código (onde 200 > 200 dá falso e cai no frete padrão), esse teste passou sem problemas.

só que a regra de negócio diz que compras a partir de "200 ou mais" deveriam ter frete grátis.

ficando claro o perigo de testar o código pelo próprio código, a função roda, mas o sistema estava violando o requisito do negócio sem ninguém perceber

# 6. Gere uma nova suíte com IA

Agora forneça à ferramenta de IA:

- a implementação;
- a especificação funcional;
- a instrução abaixo.

## Prompt

```text
Você é responsável por testes unitários de um projeto Python.

Analise a implementação e a especificação funcional fornecidas.

1. Gere uma suíte pytest derivada da especificação.
2. Inclua casos representativos, fronteiras e tratamento de erro.
3. Não altere silenciosamente a implementação.
4. Diferencie comportamento observado de comportamento esperado.
5. Para cada teste, explique brevemente de qual requisito veio a expectativa.
6. Caso encontre divergência entre código e especificação, crie um teste que a revele.

Retorne primeiro a análise e depois somente o código dos testes.
```

Cole a resposta da IA abaixo, mas **revise-a antes de executar**.


In [7]:
%%writefile test_frete_rodada2.py
# COLE AQUI a suíte produzida/revisada na Rodada 2.
#
# Ela deve usar a ESPECIFICAÇÃO como fonte principal do oracle.


import pytest
from frete import calcular_frete

# ----------------------------------------------------------------------
# Requisito 1: Valor de compra não pode ser negativo (ValueError)
# ----------------------------------------------------------------------
def test_valor_compra_negativo_deve_lançar_excecao():
    # Requisito 1: Se valor_compra < 0, deve lançar ValueError
    with pytest.raises(ValueError, match="valor_compra não pode ser negativo"):
        calcular_frete(-0.01, True)


# ----------------------------------------------------------------------
# Requisito 2: Cliente premium com R$ 200,00 ou mais tem frete grátis (0.0)
# ----------------------------------------------------------------------
def test_cliente_premium_compra_acima_de_200_frete_gratis():
    # Requisito 2: Compra > 200 para premium deve retornar 0.0
    assert calcular_frete(200.01, True) == 0.0

def test_cliente_premium_compra_exatamente_200_revela_bug():
    # Requisito 2: Compra de "R$ 200,00 ou mais" para premium deve retornar 0.0.
    # NOTA: Este teste vai FALHAR no código atual porque o código usa > 200 em vez de >= 200.
    assert calcular_frete(200.00, True) == 0.0


# ----------------------------------------------------------------------
# Requisito 3: Nos demais casos, o frete custa R$ 20,00
# ----------------------------------------------------------------------
def test_cliente_premium_compra_abaixo_de_200_frete_padrao():
    # Requisito 3: Cliente premium com compra < 200 (ex: 199.99) paga 20.0
    assert calcular_frete(199.99, True) == 20.0

def test_cliente_nao_premium_compra_qualquer_valor_frete_padrao():
    # Requisito 3: Cliente não premium paga 20.0 mesmo com compra >= 200
    assert calcular_frete(200.00, False) == 20.0
    assert calcular_frete(500.00, False) == 20.0

def test_compra_valor_zero_frete_padrao():
    # Requisito 3: Compra de 0.0 é válida (>= 0) e cai no frete padrão de 20.0
    assert calcular_frete(0.0, False) == 20.0


Writing test_frete_rodada2.py


## 7. Execute os testes derivados da especificação

Agora rode a nova suíte contra a implementação **ainda não corrigida**.


In [8]:
!python -m pytest -q test_frete_rodada2.py

..F...                                                                   [100%]
=================================== FAILURES ===================================
____________ test_cliente_premium_compra_exatamente_200_revela_bug _____________

    def test_cliente_premium_compra_exatamente_200_revela_bug():
        # Requisito 2: Compra de "R$ 200,00 ou mais" para premium deve retornar 0.0.
        # NOTA: Este teste vai FALHAR no código atual porque o código usa > 200 em vez de >= 200.
>       assert calcular_frete(200.00, True) == 0.0
E       assert 20.0 == 0.0
E        +  where 20.0 = calcular_frete(200.0, True)

test_frete_rodada2.py:28: AssertionError
=========================== short test summary info ============================
FAILED test_frete_rodada2.py::test_cliente_premium_compra_exatamente_200_revela_bug - assert 20.0 == 0.0
1 failed, 5 passed in 0.07s


## 8. Analise a falha

Se um teste falhou, não corrija imediatamente.

Primeiro responda:

1. Qual entrada provocou a divergência?

a combinação de valor_compra = 200.00 e cliente_premium = True

2. Qual resultado a implementação produziu?
a função retornou 20.0 e cobrou o frete padrão

3. Qual resultado a especificação exige?
regra de negócio exige 0.0,frete grátis para compras de R$ 200,00 ou mais.

4. A falha está no teste ou na implementação?

na implementação, o teste está correto porque seguiu o que a especificação exigia, porém código é que contém o bug.

5. Qual linha/condição do código explica a divergência?

o if cliente_premium and valor_compra > 200:

a comparação  > em vez de maior ou igual >=

por issso, o valor de 200 não entra no if do frete grátis e cai direto no return 20.0

6. Por que os testes da Rodada 1 poderiam não considerar isso um defeito?


não existia uma especificação de negócio para consultar.
a IA assumiu que o retorno 20.0 para o valor 200.00 era o comportamento comum.

**Sua análise:**

> Escreva aqui.


# 9. Teste mínimo de fronteira

Independentemente da suíte gerada pela IA, escreva manualmente **um teste mínimo** que represente exatamente a fronteira da regra de negócio.

Requisito relevante:

> Cliente premium com compra de **R$ 200,00 ou mais** tem frete grátis.

O teste deve:

- utilizar exatamente o valor de fronteira;
- usar a especificação como *oracle*;
- falhar na implementação atual caso exista divergência.


In [9]:
from frete import calcular_frete

# ESCREVA AQUI o teste mínimo de fronteira.

def test_fronteira_cliente_premium_200():
    resultado = calcular_frete(valor_compra=200.00, cliente_premium=True)
    assert resultado == 0.0


## 10. Corrija a implementação

Somente agora altere a implementação para fazê-la atender à especificação.

Faça a **menor alteração necessária**.

Evite refatorações que não sejam necessárias para corrigir o defeito.


In [ ]:
%%writefile frete.py
def calcular_frete(valor_compra: float, cliente_premium: bool) -> float:
    """
    Calcula o valor do frete de uma compra.
    """
    # CORRIJA SOMENTE O NECESSÁRIO.

    if valor_compra < 0:
        raise ValueError("valor_compra não pode ser negativo")

    if cliente_premium and valor_compra > 200:
        return 0.0

    return 20.0


In [13]:
#correção

%%writefile frete.py
def calcular_frete(valor_compra: float, cliente_premium: bool) -> float:
    """
    Calcula o valor do frete de uma compra.
    """
    if valor_compra < 0:
        raise ValueError("valor_compra não pode ser negativo")

    # >= no lugar >
    if cliente_premium and valor_compra >= 200:
        return 0.0

    return 20.0

Overwriting frete.py


## 11. Execute novamente as duas suítes

Primeiro, execute os testes derivados da especificação.


In [11]:
!python -m pytest -q test_frete_rodada2.py

......                                                                   [100%]
6 passed in 0.01s


Agora execute novamente os testes de caracterização da Rodada 1.


In [12]:
!python -m pytest -q test_frete_rodada1.py

.F....                                                                   [100%]
=================================== FAILURES ===================================
___________________ test_frete_padrao_premium_exatamente_200 ___________________

    def test_frete_padrao_premium_exatamente_200():
        # Expectativa (20.0): A condição '> 200' é falsa para 200.0, caindo no 'return 20.0'
>       assert calcular_frete(200.0, True) == 20.0
E       assert 0.0 == 20.0
E        +  where 0.0 = calcular_frete(200.0, True)

test_frete_rodada1.py:12: AssertionError
=========================== short test summary info ============================
FAILED test_frete_rodada1.py::test_frete_padrao_premium_exatamente_200 - assert 0.0 == 20.0
1 failed, 5 passed in 0.07s


## 12. Atenção: um teste antigo pode agora falhar

Se um teste da Rodada 1 esperava o comportamento antigo exatamente na fronteira, ele pode falhar depois da correção.

Isso não significa automaticamente que a correção esteja errada.

Analise:

- Qual era a fonte do oracle desse teste antigo?

o teste foi gerado olhando só para o código

- Ele registrava um requisito ou apenas o comportamento anterior?

 comportamento anterior

- Depois que uma especificação independente foi revelada, esse teste ainda representa o comportamento desejado?

não, a especificação oficial deixou claro o erro, esse teste antigo tornou-se obsoleto

**Sua resposta:**

> Escreva aqui.


# 13. Classifique os testes

Para cada teste das suas duas suítes, classifique-o como uma das opções:

- **Caracterização** — expectativa derivada do comportamento observado da implementação.
- **Requisito** — expectativa derivada da especificação funcional.
- **Erro/validação** — verifica o contrato de erro explicitamente definido.
- **Fronteira** — concentra-se em um limite de decisão relevante.

Um mesmo teste pode receber mais de uma classificação quando fizer sentido.

### Registro

| Teste | Fonte do oracle | Classificação | Deve permanecer após a correção? |
|---|---|---|---|
| **Rodada 1:** |
| `test_frete_gratis_premium_acima_200` | Comportamento observado (código) | Caracterização, Fronteira | Não |
| `test_frete_padrao_premium_exatamente_200` | Comportamento observado (código) | Caracterização, Fronteira | Não |
| `test_frete_padrao_premium_abaixo_200` | Comportamento observado (código) | Caracterização | Não |
| `test_frete_padrao_nao_premium_acima_200` | Comportamento observado (código) | Caracterização | Não |
| `test_frete_padrao_limite_zero` | Comportamento observado (código) | Caracterização, Fronteira | Não |
| `test_erro_valor_negativo` | Comportamento observado (código), Requisito 1 | Caracterização, Erro/validação, Requisito, Fronteira | Sim |
| **Rodada 2:** |
| `test_valor_compra_negativo_deve_lançar_excecao` | Requisito 1 | Requisito, Erro/validação, Fronteira | Sim |
| `test_cliente_premium_compra_acima_de_200_frete_gratis` | Requisito 2 | Requisito, Fronteira | Sim |
| `test_cliente_premium_compra_exatamente_200_revela_bug` | Requisito 2 | Requisito, Fronteira | Sim |
| `test_cliente_premium_compra_abaixo_de_200_frete_padrao` | Requisito 3 | Requisito | Sim |
| `test_cliente_nao_premium_compra_qualquer_valor_frete_padrao` | Requisito 3 | Requisito | Sim |
| `test_compra_valor_zero_frete_padrao` | Requisito 3 | Requisito, Fronteira | Sim |
| **Teste mínimo de fronteira (Seção 9):** |
| `test_fronteira_cliente_premium_200` | Requisito 2 | Requisito, Fronteira | Sim |


# 14. Desafio para a IA

Pergunte à IA:

```text
Todos os testes da Rodada 1 passavam antes de a especificação ser fornecida.
Explique por que isso não era evidência suficiente de correção funcional.

Use os conceitos:
- test oracle;
- teste de caracterização;
- requisito;
- condição de fronteira;
- defeito de implementação.

Não invente regras além da especificação fornecida.
```

Depois, avalie criticamente a resposta.

### Verificação humana

Marque:

- [x] A IA distinguiu comportamento observado de comportamento correto.
- [x] A IA identificou que um teste pode cristalizar um defeito.
- [x] A IA explicou corretamente o papel do *oracle*.
- [x] A IA não criou regras inexistentes.
- [x] A IA identificou a importância da fronteira.


# 15. Conclusão individual

Responda com suas próprias palavras:

### A.
Por que uma suíte 100% verde pode coexistir com um defeito de negócio?

> Resposta:  teste verde só garante que a resposta do código bateu com a expectativa que foi programada, se a expectativa esperava um valor errado por causa de um bug, o teste vai passsar do mesmo jeito.

### B.
Qual foi a diferença entre o *oracle* usado na Rodada 1 e o usado na Rodada 2?

> Resposta: na rodada 1, a expectativa veio da leitura do próprio código, agora,na rodada 2, veio da regra de negócio oficial.

### C.
Por que testar exatamente os valores de fronteira é importante?

> Resposta: porque uma simples troca de > por >= muda a regra no valor exato do limite.

### D.
Qual risco existe ao pedir para uma IA "gerar testes para este código" quando não fornecemos requisitos?  

> Resposta: a IA vai achar que o código atual é a verdade absoluta, criando testes que validam e os próprios bugs presentes na implementação.

### E.
Depois desta atividade, complete:

> **Um teste automatizado não sabe sozinho o que é correto porque...**



Resposta: ...ele apenas compara dois valores sem entender o contexto, se ensinarmos o valor errado ao teste, ele vai aceitar o erro como certo.

> Resposta:


# 16. Entrega

Entregue:

1. este notebook preenchido;
2. `test_frete_rodada1.py`;
3. `test_frete_rodada2.py`;
4. a implementação final de `frete.py`;
5. suas respostas às análises.

## Critério essencial

Não será avaliado apenas se os testes passam.

Será avaliado se você consegue justificar **de onde veio a expectativa de cada teste** e distinguir:

> **“o código faz isso”**  
> de  
> **“o sistema deveria fazer isso”.**
